# Sentiment Analysis on Employee Survey Responses

This notebook walks through the NLP-based sentiment analysis module of the People Analytics suite.

**What we cover:**
1. Load and explore survey response data
2. Run VADER-inspired sentiment scoring on free-text comments
3. Aggregate sentiment by department, quarter, and level
4. Extract themes using LDA topic modeling
5. Track sentiment trends and detect shifts
6. Flag concerning responses for HR follow-up

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['figure.figsize'] = (10, 5)

from src.sentiment.survey_analyzer import SurveyAnalyzer, VADERSentiment
from src.sentiment.theme_extractor import ThemeExtractor
from src.sentiment.trend_tracker import TrendTracker

## 1. Load Survey Data

In [ ]:
surveys = pd.read_csv('../data/survey_responses.csv')
print(f'Survey responses: {len(surveys):,}')
print(f'Quarters: {surveys["survey_quarter"].nunique()}')
print(f'Departments: {surveys["department"].nunique()}')
surveys.head()

In [ ]:
# Distribution of Likert scores
score_cols = ['engagement_score', 'manager_score', 'growth_score', 'worklife_score', 'belonging_score']
fig, axes = plt.subplots(1, 5, figsize=(16, 3), sharey=True)
for ax, col in zip(axes, score_cols):
    surveys[col].value_counts().sort_index().plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title(col.replace('_score', '').title(), fontsize=9)
    ax.set_xlabel('')
axes[0].set_ylabel('Count')
plt.suptitle('Likert Score Distributions', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## 2. Sentiment Scoring

The `VADERSentiment` engine uses a curated HR/workplace lexicon with negation handling, intensifiers, and punctuation amplification.

In [ ]:
analyzer = SurveyAnalyzer()
analyzed = analyzer.analyze_responses(surveys)
print(f'Sentiment distribution:')
print(analyzed['sentiment_label'].value_counts())
print(f'\nMean compound score: {analyzed["sentiment_compound"].mean():.3f}')

In [ ]:
# Compound score distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
analyzed['sentiment_compound'].hist(bins=40, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].axvline(0.2, color='green', linestyle='--', alpha=0.7, label='Positive threshold')
axes[0].axvline(-0.2, color='red', linestyle='--', alpha=0.7, label='Negative threshold')
axes[0].set_title('Compound Sentiment Score Distribution')
axes[0].set_xlabel('Compound Score')
axes[0].legend(fontsize=8)

# Label pie chart
colors = {'positive': '#2ecc71', 'neutral': '#95a5a6', 'negative': '#e74c3c'}
label_counts = analyzed['sentiment_label'].value_counts()
axes[1].pie(label_counts.values, labels=label_counts.index,
            colors=[colors[l] for l in label_counts.index],
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('Sentiment Label Distribution')

plt.tight_layout()
plt.show()

## 3. Department-Level Aggregation

In [ ]:
dept_sentiment = analyzer.aggregate_sentiment(analyzed, group_by='department')
dept_sentiment

In [ ]:
# Department sentiment comparison
fig, ax = plt.subplots(figsize=(10, 5))
dept_sorted = dept_sentiment.sort_values('mean_sentiment')
colors_bar = ['#e74c3c' if v < 0 else '#2ecc71' for v in dept_sorted['mean_sentiment']]
dept_sorted['mean_sentiment'].plot(kind='barh', ax=ax, color=colors_bar)
ax.set_xlabel('Mean Compound Sentiment')
ax.set_title('Mean Sentiment by Department')
ax.axvline(0, color='black', linewidth=0.5)
for i, (idx, row) in enumerate(dept_sorted.iterrows()):
    ax.text(row['mean_sentiment'] + 0.01, i, f'{row["pct_negative"]:.0%} neg',
            va='center', fontsize=8, color='#e74c3c')
plt.tight_layout()
plt.show()

In [ ]:
# Sentiment heatmap: department x quarter
pivot = analyzed.groupby(['department', 'survey_quarter'])['sentiment_compound'].mean().unstack()
fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(pivot.values, cmap='RdYlGn', aspect='auto', vmin=-0.5, vmax=0.5)
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=9)
plt.colorbar(im, label='Mean Sentiment')
ax.set_title('Sentiment Heatmap: Department x Quarter')
plt.tight_layout()
plt.show()

## 4. Theme Extraction (LDA Topic Modeling)

LDA discovers latent themes in free-text comments that Likert scores alone cannot reveal.

In [ ]:
# Filter to non-empty comments
texts = analyzed[analyzed['free_text_comment'].str.len() > 5]['free_text_comment'].tolist()
print(f'Comments for topic modeling: {len(texts):,}')

extractor = ThemeExtractor(n_topics=6)
extractor.fit(texts)
theme_summary = extractor.summary()
theme_summary

In [ ]:
# Theme prevalence bar chart
fig, ax = plt.subplots(figsize=(10, 4))
theme_summary_sorted = theme_summary.sort_values('prevalence', ascending=True)
ax.barh(theme_summary_sorted['label'], theme_summary_sorted['prevalence'],
        color='steelblue', edgecolor='white')
ax.set_xlabel('Prevalence (fraction of responses)')
ax.set_title('Discovered Themes in Employee Survey Comments')
for i, (_, row) in enumerate(theme_summary_sorted.iterrows()):
    ax.text(row['prevalence'] + 0.005, i, row['top_words'][:40],
            va='center', fontsize=7, color='gray')
plt.tight_layout()
plt.show()

In [ ]:
# Theme prevalence by department
theme_by_dept = extractor.theme_by_group(analyzed[analyzed['free_text_comment'].str.len() > 5],
                                          text_column='free_text_comment',
                                          group_column='department')
fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(theme_by_dept.values, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(len(theme_by_dept.columns)))
ax.set_xticklabels([c.split('_', 2)[-1][:20] for c in theme_by_dept.columns],
                    rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(theme_by_dept.index)))
ax.set_yticklabels(theme_by_dept.index, fontsize=9)
plt.colorbar(im, label='Topic Weight')
ax.set_title('Theme Prevalence by Department')
plt.tight_layout()
plt.show()

## 5. Sentiment Trend Tracking

In [ ]:
tracker = TrendTracker()
report = tracker.summary_report(analyzed)
print(f'Periods analyzed: {report["periods_analyzed"]}')
print(f'Total alerts: {report["total_alerts"]}')
print(f'Significant alerts: {report["significant_alerts"]}')
print(f'Declining groups: {report["declining_groups"]}')
print(f'Improving groups: {report["improving_groups"]}')

In [ ]:
# Sentiment trend lines by department
trends = report['trends']
fig, ax = plt.subplots(figsize=(10, 5))
for dept in trends['group'].unique():
    dept_data = trends[trends['group'] == dept]
    ax.plot(dept_data['period'], dept_data['mean_sentiment'], marker='o', label=dept, linewidth=1.5)
ax.set_xlabel('Survey Period')
ax.set_ylabel('Mean Sentiment')
ax.set_title('Sentiment Trends by Department')
ax.legend(fontsize=7, ncol=2, loc='upper left', bbox_to_anchor=(1, 1))
ax.axhline(0, color='gray', linestyle=':', linewidth=0.5)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Alert details
if report['alerts']:
    alert_df = pd.DataFrame([{
        'Department': a.group,
        'From': a.period_from,
        'To': a.period_to,
        'Change': f'{a.change:+.3f}',
        'Direction': a.direction,
        'Significance': a.significance
    } for a in report['alerts']])
    print('Sentiment Shift Alerts:')
    display(alert_df)
else:
    print('No significant sentiment shifts detected.')

In [ ]:
# Diverging groups
diverging = report['diverging_groups']
if not diverging.empty:
    fig, ax = plt.subplots(figsize=(8, 4))
    colors_div = ['#e74c3c' if d == 'diverging_negative' else '#2ecc71' if d == 'diverging_positive' else '#95a5a6'
                  for d in diverging['direction']]
    ax.barh(diverging['group'], diverging['trend_slope'], color=colors_div)
    ax.set_xlabel('Trend Slope (deviation from org mean)')
    ax.set_title('Diverging Departments')
    ax.axvline(0, color='black', linewidth=0.5)
    plt.tight_layout()
    plt.show()

## 6. Concerning Responses (HR Follow-up Queue)

In [ ]:
concerning = analyzer.flag_concerning_responses(analyzed, threshold=-0.3)
print(f'Responses flagged for follow-up: {len(concerning):,}')
print(f'\nDepartment breakdown:')
print(concerning['department'].value_counts())
print(f'\nSample concerning comments:')
for _, row in concerning.head(5).iterrows():
    print(f'  [{row["department"]}] (score: {row["sentiment_compound"]:.2f}) {row["free_text_comment"][:80]}')

## Key Takeaways

- **VADER-based sentiment** provides immediate, interpretable scoring without API keys
- **Theme extraction** surfaces concerns invisible in Likert scores (e.g., "tools" complaints in Ops vs. praise in Eng)
- **Trend tracking** detects statistically significant shifts before they become crises
- **Production path:** swap `VADERSentiment` for `AzureOpenAISentiment` to leverage GPT-4 for nuanced analysis